In [ ]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

BASE = "http://localhost:8080"
original = "https://practicum.yandex.ru/"

r = requests.get(f"{BASE}/health", timeout=5)
print("health:", r.status_code, r.text)

token = requests.post(
    os.environ["KEYCLOAK_TOKEN_URL"],
    data={
        "grant_type": "client_credentials",
        "client_id": os.environ["KEYCLOAK_DEBUG_CLIENT_ID"],
        "client_secret": os.environ["KEYCLOAK_DEBUG_CLIENT_SECRET"],
    },
    timeout=5,
)
token.raise_for_status()
access_token = token.json()["access_token"]

r = requests.post(
    f"{BASE}/api/shorten",
    json={"url": original},
    headers={"Authorization": f"Bearer {access_token}"},
    timeout=5,
)
print("shorten:", r.status_code, r.json())
short_url = r.json()["result"]
short_id = short_url.rsplit("/", 1)[-1]

r = requests.get(f"{BASE}/{short_id}", allow_redirects=False, timeout=5)
print("redirect:", r.status_code, r.headers.get("Location"))
